# Hour 3 · Divination as decision trees


**Goal:** formalize an omen text's *if → then* logic as data and visualize it as a tree — then discuss where LLMs help and where they mislead.

We use a real Ugaritic **birth-omen** text (the KTU 1.103 “malformed-foetus” series) and a hand-built decision tree of it (`data/omens/`).

**Needs:** `pip install networkx`.

*Reading:* `docs/07-divination.md`.

## 0. Setup


In [ ]:
# === SETUP — run me first ===
# Makes the bundled Copenhagen Ugaritic Corpus (CUC) available. No edits needed.
import sys
sys.path.append("..")                       # so Python can find data/loader.py
from data.loader import load_texts

texts = load_texts()                        # 278 real KTU tablets, offline
print(f"Loaded {len(texts)} tablets.")
texts[0]["ktu"], texts[0]["genre"], texts[0]["lines"][:2]

## 1. The omen text (English working translation)
Note the strict conditional pattern: *if the foetus lacks X → consequence for land/king/cattle*.

In [ ]:
import pathlib
txt = pathlib.Path("../data/omens/ugaritic_birth_omens.txt").read_text(encoding="utf-8")
print(txt[:900])

## 2. The same logic, formalized as a tree (nested JSON)
This is the **philologist's** modelling step: deciding the observed sign → variant → outcome.

In [ ]:
from data.loader import load_omen_tree
tree = load_omen_tree()
import json
print(json.dumps(tree, indent=2, ensure_ascii=False)[:700], "...")

## 3. Turn the nested dict into a graph

In [ ]:
import networkx as nx
G = nx.DiGraph()
counter = [0]
def add(node_label, subtree, parent=None):
    nid = counter[0]; counter[0]+=1
    label = str(node_label)
    G.add_node(nid, label=label, leaf=not isinstance(subtree, dict))
    if parent is not None: G.add_edge(parent, nid)
    if isinstance(subtree, dict):
        for k, v in subtree.items():
            add(k, v, nid)
    else:
        lid = counter[0]; counter[0]+=1
        G.add_node(lid, label="→ "+str(subtree)[:40], leaf=True)
        G.add_edge(nid, lid)
add("omen", tree["omen"])
print("nodes:", G.number_of_nodes())

## 4. Draw the decision tree
A simple left-to-right layered layout (no extra deps).

In [ ]:
import matplotlib.pyplot as plt
def layered_pos(G, root=0):
    import collections
    depth={root:0}; order=collections.deque([root])
    while order:
        u=order.popleft()
        for v in G.successors(u):
            depth[v]=depth[u]+1; order.append(v)
    rows=collections.defaultdict(list)
    for n,d in depth.items(): rows[d].append(n)
    pos={}
    for d,ns in rows.items():
        for i,n in enumerate(ns):
            pos[n]=(d, -i*1.0/max(1,len(ns)-0 ) - i)
    return pos
pos = layered_pos(G)
labels = {n:G.nodes[n]["label"] for n in G.nodes()}
colors = ["#ffe2b8" if not G.nodes[n]["leaf"] else "#d9f2d9" for n in G.nodes()]
plt.figure(figsize=(13,11))
nx.draw(G, pos, labels=labels, node_color=colors, node_size=900, font_size=7,
        arrows=True, edge_color="#aaa")
plt.title("Sheep-birth omen series as a decision tree"); plt.axis("off"); plt.show()

## 5. Where LLMs help — and where they are dangerous
**Help:** extract this structure from raw text, validate the JSON, compare two omen trees, generate a plain-language explanation.

**Danger:** *invent* missing branches (the text is full of `[...]` gaps), *smooth over* ambiguity, *lose* philological detail.

> **Try it:** paste the omen text into **UgaritGPT** (see `docs/00-resources.md`) and ask it to produce this JSON tree — then check what it filled in that the tablet does **not** actually say. That gap is the lesson: philology + data modelling + LLM, with a human in the loop.